In [ ]:
!pip install -q "docling[ocrmac]"

# Docling vs Grobid

Other alternatives - unstructured.io, llamaparse, pymupdf

## Parsing the paper with Docling

In [ ]:
from docling.document_converter import DocumentConverter

source = "https://arxiv.org/pdf/2201.00689"  # file path or URL
converter = DocumentConverter()
doc = converter.convert(source).document

print(doc.export_to_markdown())  # output: "### Docling Technical Report[...]"

## Parse the same paper with GROBID

[GROBID](https://github.com/kermitt2/grobid) is a service specialized in extracting structure from **scholarly** PDFs. It runs as a local server and returns **TEI XML**. Here we send the *same* paper (`source`) to a local GROBID instance and compare its output with Docling's.

Start the server first (CRF image, ~4 GB RAM):

```bash
docker run --rm --init --ulimit core=0 -p 8070:8070 grobid/grobid:0.9.0-crf
```

In [ ]:
!pip install -q requests lxml

In [ ]:
import requests

GROBID = "http://localhost:8070"

# Sanity check: the local GROBID server must be running
# (docker run --rm --init -p 8070:8070 grobid/grobid:0.9.0-crf)
assert requests.get(f"{GROBID}/api/isalive", timeout=10).text == "true", (
    "GROBID is not alive on :8070"
)

# GROBID needs the raw PDF bytes. Reuse the same `source` URL Docling used.
pdf_bytes = requests.get(source, headers={"User-Agent": "Mozilla/5.0"}, timeout=60).content
print(f"Downloaded PDF: {len(pdf_bytes):,} bytes")

# processFulltextDocument -> full TEI XML (header + body + references)
resp = requests.post(
    f"{GROBID}/api/processFulltextDocument",
    files={"input": ("paper.pdf", pdf_bytes, "application/pdf")},
    data={"consolidateHeader": "0", "consolidateCitations": "0"},
    timeout=300,
)
resp.raise_for_status()
tei = resp.text
print(f"TEI XML: {len(tei):,} chars")

In [ ]:
from lxml import etree

# GROBID returns TEI XML; everything lives under the TEI namespace.
TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}
root = etree.fromstring(tei.encode("utf-8"))


def text_of(el):
    """Flatten an element's text content; '' if the element is missing."""
    if el is None:
        return ""
    return " ".join(" ".join(el.itertext()).split())


# Build a Markdown-ish view from the TEI so it's comparable to Docling's output.
lines = []

title = root.find(".//tei:titleStmt/tei:title", TEI_NS)
if text_of(title):
    lines += [f"## {text_of(title)}", ""]

abstract = root.find(".//tei:profileDesc/tei:abstract", TEI_NS)
if text_of(abstract):
    lines += ["## Abstract", "", text_of(abstract), ""]

for div in root.findall(".//tei:body/tei:div", TEI_NS):
    head = div.find("tei:head", TEI_NS)
    if text_of(head):
        lines += [f"## {text_of(head)}", ""]
    for p in div.findall("tei:p", TEI_NS):
        if text_of(p):
            lines += [text_of(p), ""]

grobid_md = "\n".join(lines)
print(grobid_md[:2000])

## Compare: Docling vs GROBID

Both produce the body text, but they differ in character:

- **Docling** — ML layout + table models. Great Markdown structure (headings, tables, reading order), figures located. General-purpose PDF.
- **GROBID** — CRF models specialized for *scholarly* PDFs. Weaker generic Markdown, but rich structured bibliographic metadata (authors, affiliations, parsed references) via TEI XML.

In [ ]:
docling_md = doc.export_to_markdown()


def stats(name, md):
    print(
        f"{name:8} | chars: {len(md):>8,} | words: {len(md.split()):>7,} | lines: {md.count(chr(10)):>5}"
    )


print("Output size comparison")
print("-" * 50)
stats("Docling", docling_md)
stats("GROBID", grobid_md)

# Structured metadata GROBID gives you for free (Docling does not expose these directly)
n_refs = len(root.findall(".//tei:listBibl/tei:biblStruct", TEI_NS))
n_authors = len(root.findall(".//tei:sourceDesc//tei:author/tei:persName", TEI_NS))
print("\nGROBID structured extraction")
print("-" * 50)
print(f"authors parsed   : {n_authors}")
print(f"references parsed: {n_refs}")

# Side-by-side preview of the first lines of each output
print("\nFirst 400 chars\n" + "=" * 50)
print("[DOCLING]\n" + docling_md[:400])
print("\n[GROBID]\n" + grobid_md[:400])

In [ ]:
print(grobid_md)

print(doc.export_to_markdown())

Grobid better parses the research paper, particularly, preserving the structure and right latin/greek mathematical symbols.

# Inline image parsing with VLM

In [1]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [6]:
import os

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, PictureDescriptionApiOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

pipeline_options = PdfPipelineOptions()
pipeline_options.enable_remote_services = True
pipeline_options.generate_picture_images = True
pipeline_options.do_picture_description = True

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_MODEL = (
    "qwen/qwen3-vl-8b-instruct"  # any vision-capable model slug from openrouter.ai/models
)

pipeline_options.picture_description_options = PictureDescriptionApiOptions(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "X-Title": "Docling Picture Description",
    },
    params={"model": OPENROUTER_MODEL, "max_tokens": 500, "temperature": 0.1},
    prompt=(
        "Analyze this image extracted from a document. "
        "If it is a table, transcribe its contents markdown format. "
        "If it is a chart or figure, provide a highly detailed, dense text summary of its structural data."
    ),
    timeout=120,
    concurrency=1,
)

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

result = converter.convert("https://arxiv.org/pdf/2201.00689")
print(result.document.export_to_markdown())

Loading weights: 100%|██████████| 770/770 [00:04<00:00, 190.22it/s]


## CausalMTA: Eliminating the User Confounding Bias for Causal Multi-touch Attribution

Di Yao ∗

Institute of Computing Technology, Chinese Academy of Sciences China yaodi@ict.ac.cn Chang Gong Institute of Computing Technology, Chinese Academy of Sciences University of Chinese Academy of Sciences China gongchang21z@ict.ac.cn

## Sheng Chen

Strategic Data Solutions (SDS) Group, Alibaba Inc China chensheng.cs@alibaba-inc.com

## Lei Zhang

Strategic Data Solutions (SDS) Group,

Alibaba Inc China zl165646@alibaba-inc.com

## Jingping Bi ∗

Institute of Computing Technology, Chinese Academy of Sciences China bjp@ict.ac.cn

## KEYWORDS

multi-touch attribution, counterfactual prediction, computational advertising

## ACMReference Format:

Di Yao, Chang Gong, Lei Zhang, Sheng Chen, and Jingping Bi. 2022. CausalMTA: Eliminating the User Confounding Bias for Causal Multi-touch Attribution. In Proceedings of the 28th ACM SIGKDD Conference on Knowledge Discovery and Data Mining (KDD '22), Augu

# Parsing confidence

In [22]:
import statistics
from collections import defaultdict

# ── 1. Top-level conversion status ───────────────────────────────────────────
print(f"Status : {result.status}")  # SUCCESS / PARTIAL_SUCCESS / FAILURE
print(f"Errors : {result.errors}")  # list of ErrorItem (page_no + message)

# ── 2. Per-element layout confidence (from DocLayNet layout model) ────────────
confidence_rows = []

for page in result.pages:
    page_no = page.page_no

    layout = page.predictions.layout if page.predictions else None
    if not layout:
        continue

    for cluster in layout.clusters:
        confidence_rows.append(
            {
                "page": page_no,
                "label": cluster.label.value,  # e.g. TEXT, TABLE, PICTURE, SECTION_HEADER
                "confidence": cluster.confidence,
                "bbox": cluster.bbox,
            }
        )

# ── 3. Summary by element type ────────────────────────────────────────────────
print("\n── Confidence by element type ──")
by_label = defaultdict(list)
for row in confidence_rows:
    by_label[row["label"]].append(row["confidence"])

for label, scores in sorted(by_label.items()):
    print(
        f"  {label:<25} n={len(scores):>3}  "
        f"mean={statistics.mean(scores):.3f}  "
        f"min={min(scores):.3f}  "
        f"max={max(scores):.3f}"
    )

# ── 4. Flag low-confidence elements (tune threshold per doc type) ─────────────
THRESHOLD = 0.6
low_conf = [r for r in confidence_rows if r["confidence"] < THRESHOLD]

if low_conf:
    print(f"\n── Low-confidence elements (< {THRESHOLD}) ──")
    for r in low_conf:
        print(
            f"  Page {r['page']:>2}  {r['label']:<25}  conf={r['confidence']:.3f}  bbox={r['bbox']}"
        )
else:
    print("\nAll elements above confidence threshold ✓")

# ── 5. Overall document-level confidence score (weighted mean) ────────────────
all_scores = [r["confidence"] for r in confidence_rows]
doc_confidence = statistics.mean(all_scores) if all_scores else 0.0
print(f"\nDocument-level confidence (weighted mean): {doc_confidence:.3f}")

# ── 6. Per-page confidence (useful for flagging bad pages) ────────────────────
print("\n── Per-page confidence ──")
page_scores = defaultdict(list)
for r in confidence_rows:
    page_scores[r["page"]].append(r["confidence"])

for pg, scores in sorted(page_scores.items()):
    mean_pg = statistics.mean(scores)
    flag = " ⚠️" if mean_pg < THRESHOLD else ""
    print(f"  Page {pg:>2}: mean={mean_pg:.3f}  elements={len(scores)}{flag}")

# ── 7. OCR cell-level confidence (only when OCR backend is active) ────────────
print("\n── OCR cell confidence (sampled) ──")
for page in result.pages:
    for cell in page.cells:
        conf = getattr(cell, "confidence", None)
        if conf is not None:
            print(
                f"  Page {page.page_no}  cell_id={cell.id}  ocr_conf={conf:.3f}  text={cell.text[:40]!r}"
            )

Status : ConversionStatus.SUCCESS
Errors : []

── Confidence by element type ──
  caption                   n= 10  mean=0.943  min=0.923  max=0.975
  footnote                  n=  1  mean=0.848  min=0.848  max=0.848
  formula                   n= 23  mean=0.903  min=0.501  max=0.974
  key_value_region          n=  1  mean=0.552  min=0.552  max=0.552
  list_item                 n= 71  mean=0.895  min=0.556  max=0.980
  page_header               n= 21  mean=0.920  min=0.880  max=0.941
  picture                   n=  6  mean=0.928  min=0.715  max=0.976
  section_header            n= 34  mean=0.891  min=0.615  max=0.954
  table                     n=  5  mean=0.983  min=0.979  max=0.987
  text                      n=127  mean=0.926  min=0.503  max=1.000

── Low-confidence elements (< 0.6) ──
  Page  1  text                       conf=0.537  bbox=l=394.577 t=142.82297960000005 r=547.8135040175999 b=147.3659252 coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>
  Page  1  key_value_region        

In [25]:
low_conf

[{'page': 1,
  'label': 'text',
  'confidence': 0.5367892384529114,
  'bbox': BoundingBox(l=394.577, t=142.82297960000005, r=547.8135040175999, b=147.3659252, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>)},
 {'page': 1,
  'label': 'key_value_region',
  'confidence': 0.5522691011428833,
  'bbox': BoundingBox(l=52.90776824951172, t=701.7467651367188, r=166.88525390625, b=709.1648559570312, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>)},
 {'page': 5,
  'label': 'list_item',
  'confidence': 0.5777231454849243,
  'bbox': BoundingBox(l=59.669000000000004, t=255.88442239999983, r=98.50459040000001, b=260.04483199999993, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>)},
 {'page': 5,
  'label': 'list_item',
  'confidence': 0.5557041168212891,
  'bbox': BoundingBox(l=56.426, t=332.5964224, r=98.50459040000001, b=336.756832, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>)},
 {'page': 5,
  'label': 'list_item',
  'confidence': 0.5672545433044434,
  'bbox': BoundingBox(l=69.041, t=352.2113755, r=21

In [10]:
page = result.pages[0]
page

Page(page_no=1, size=Size(width=612.0, height=792.0), predictions=PagePredictions(layout=LayoutPrediction(clusters=[Cluster(id=6, label=<DocItemLabel.SECTION_HEADER: 'section_header'>, bbox=BoundingBox(l=60.611, t=84.2284052, r=551.3876231999999, b=119.41464614430015, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), confidence=0.9486823678016663, cells=[PdfTextCell(index=0, rgba=ColorRGBA(r=0, g=0, b=0, a=255), rect=BoundingRectangle(r_x0=60.611, r_y0=99.4896461443002, r_x1=551.3876231999999, r_y1=99.4896461443002, r_x2=551.3876231999999, r_y2=84.2284052, r_x3=60.611, r_y3=84.2284052, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), text='CausalMTA: Eliminating the User Confounding Bias for Causal', orig='CausalMTA: Eliminating the User Confounding Bias for Causal', text_direction=<TextDirection.LEFT_TO_RIGHT: 'left_to_right'>, confidence=1.0, from_ocr=False, rendering_mode=<PdfCellRenderingMode.UNKNOWN: -1>, widget=False, font_key='/F183', font_name='/UMPRKK+LinBiolinumTB'), PdfTextCell

In [14]:
import json

page_json = json.loads(page.json())

/var/folders/k5/l531tc5j2070y5w0jlhbhfcw0000gn/T/ipykernel_64401/321680676.py:2: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  page_json = json.loads(page.json())


In [16]:
page_json.keys()

dict_keys(['page_no', 'size', 'predictions', 'assembled', 'parsed_page'])

In [ ]:
page_json["predictions"]  # Raw outputs of object detection model

{'layout': {'clusters': [{'id': 6,
    'label': 'section_header',
    'bbox': {'l': 60.611,
     't': 84.2284052,
     'r': 551.3876231999999,
     'b': 119.41464614430015,
     'coord_origin': 'TOPLEFT'},
    'confidence': 0.9486823678016663,
    'cells': [{'index': 0,
      'rgba': {'r': 0, 'g': 0, 'b': 0, 'a': 255},
      'rect': {'r_x0': 60.611,
       'r_y0': 99.4896461443002,
       'r_x1': 551.3876231999999,
       'r_y1': 99.4896461443002,
       'r_x2': 551.3876231999999,
       'r_y2': 84.2284052,
       'r_x3': 60.611,
       'r_y3': 84.2284052,
       'coord_origin': 'TOPLEFT'},
      'text': 'CausalMTA: Eliminating the User Confounding Bias for Causal',
      'orig': 'CausalMTA: Eliminating the User Confounding Bias for Causal',
      'text_direction': 'left_to_right',
      'confidence': 1.0,
      'from_ocr': False},
     {'index': 1,
      'rgba': {'r': 0, 'g': 0, 'b': 0, 'a': 255},
      'rect': {'r_x0': 213.054,
       'r_y0': 119.41464614430015,
       'r_x1': 398.94

In [ ]:
page_json["assembled"]  # Ordered layout of clusters as per PDF document

{'elements': [{'label': 'section_header',
   'id': 6,
   'page_no': 1,
   'cluster': {'id': 6,
    'label': 'section_header',
    'bbox': {'l': 60.611,
     't': 84.2284052,
     'r': 551.3876231999999,
     'b': 119.41464614430015,
     'coord_origin': 'TOPLEFT'},
    'confidence': 0.9486823678016663,
    'cells': [{'index': 0,
      'rgba': {'r': 0, 'g': 0, 'b': 0, 'a': 255},
      'rect': {'r_x0': 60.611,
       'r_y0': 99.4896461443002,
       'r_x1': 551.3876231999999,
       'r_y1': 99.4896461443002,
       'r_x2': 551.3876231999999,
       'r_y2': 84.2284052,
       'r_x3': 60.611,
       'r_y3': 84.2284052,
       'coord_origin': 'TOPLEFT'},
      'text': 'CausalMTA: Eliminating the User Confounding Bias for Causal',
      'orig': 'CausalMTA: Eliminating the User Confounding Bias for Causal',
      'text_direction': 'left_to_right',
      'confidence': 1.0,
      'from_ocr': False},
     {'index': 1,
      'rgba': {'r': 0, 'g': 0, 'b': 0, 'a': 255},
      'rect': {'r_x0': 213.0

# Re-scoring the low confidence areas with a LLM

In [ ]:
import base64
import io
import os
import time
from dataclasses import dataclass, field
from pathlib import Path

import requests
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter
from docling_core.types.doc import BoundingBox, CoordOrigin
from dotenv import load_dotenv
from PIL import Image

load_dotenv("../../.env")
converter = DocumentConverter()
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
VISION_MODEL = "qwen/qwen3-vl-8b-instruct"  # swap to claude/gpt-4o etc.
SCALE = 2.0  # render resolution: 2x= 144 DPI — better for dense text/tables
PADDING_PT = 6.0  # padding in PDF points added around each bbox before crop
CONF_THRESHOLD = 0.70
RETRY_ATTEMPTS = 2
RETRY_DELAY_S = 3

In [10]:
import tempfile
import urllib.request

PDF_SOURCE = "https://arxiv.org/pdf/2201.00689"

# ── Download once ─────────────────────────────────────────────────────────────
with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
    tmp_path = Path(tmp.name)
urllib.request.urlretrieve(PDF_SOURCE, tmp_path)

# ── Convert ───────────────────────────────────────────────────────────────────
result = converter.convert(tmp_path)

# ── Open backend — reuse result.input, no new InputDocument needed ────────────
backend = PyPdfiumDocumentBackend(
    in_doc=result.input,
    path_or_stream=tmp_path,
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Label-aware prompts
#          Specialised instructions extract far better content than a generic prompt
# ─────────────────────────────────────────────────────────────────────────────
PROMPT_MAP = {
    "TABLE": (
        "This is a table extracted from a PDF. "
        "Transcribe its FULL contents in Markdown table format. "
        "Preserve all rows, columns, and header hierarchy. "
        "For merged cells, repeat the header value in each row it spans. "
        "Do not summarise — include every cell."
    ),
    "PICTURE": (
        "This is a figure or chart from a document. Provide a dense, structured description: "
        "1) Chart type and title. "
        "2) Axis labels and units. "
        "3) Key data points, trends, or notable values. "
        "4) Any legend entries. "
        "5) Caption text if visible."
    ),
    "FORMULA": (
        "This is a mathematical formula from a document. "
        "Transcribe it in LaTeX enclosed in $$...$$. "
        "If multiple equations are visible, transcribe each on its own line."
    ),
    "CODE": (
        "This is a code block from a document. "
        "Transcribe it character-perfectly inside a fenced code block, "
        "preserving all indentation and line breaks. "
        "Infer and declare the programming language."
    ),
    "TEXT": (
        "This is a text region from a document that was flagged as low confidence. "
        "Transcribe every word exactly as it appears, preserving paragraph structure. "
        "Do not paraphrase or summarise."
    ),
    "SECTION_HEADER": (
        "This is a section heading from a document. "
        "Transcribe the heading text exactly, including any numbering (e.g. '3.2 Experimental Setup'). "
        "Do not include surrounding body text."
    ),
    "CAPTION": (
        "This is a figure or table caption. "
        "Transcribe it in full, including the figure/table number and all descriptive text. "
        "Example format: 'Figure 3: Comparison of accuracy across models.'"
    ),
    "LIST_ITEM": (
        "This is a list or enumeration from a document. "
        "Transcribe each item on its own line, preserving bullet symbols, "
        "numbering, and indentation level."
    ),
    "FOOTNOTE": (
        "This is a footnote from a document. "
        "Transcribe its full text including the footnote marker number or symbol."
    ),
    "PAGE_HEADER": (
        "This is a running page header. "
        "Transcribe the text exactly as it appears — typically a document title, "
        "chapter name, or author list."
    ),
    "PAGE_FOOTER": (
        "This is a page footer. "
        "Transcribe the text exactly — typically a page number, date, or document identifier."
    ),
}
DEFAULT_PROMPT = (
    "Transcribe all content visible in this document region exactly as it appears. "
    "Preserve structure, formatting cues, and any numerical data."
)

In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Crop helper using the PDF backend
#          get_page_image(cropbox=bbox) handles coord-origin internally —
#          no manual BOTTOMLEFT→TOPLEFT math needed
# ─────────────────────────────────────────────────────────────────────────────
# ── Build page size lookup once from the result ───────────────────────────────
# Avoids loading each page backend twice just to get dimensions
page_size_map = {p.page_no: p.size for p in result.pages}


def crop_cluster_from_backend(
    backend: PyPdfiumDocumentBackend,
    page_no: int,  # 1-indexed
    bbox: BoundingBox,
    page_size,  # from page_size_map[page_no]
    scale: float = 2.0,
    padding_pt: float = 6.0,
) -> Image.Image:
    page_w: float = page_size.width
    page_h: float = page_size.height

    # ── 1. Add padding in the correct direction for this origin ───────────────
    if bbox.coord_origin == CoordOrigin.TOPLEFT:
        # TOPLEFT: t < b — y increases downward
        # expand outward: shrink t, grow b
        padded = BoundingBox(
            l=bbox.l - padding_pt,
            r=bbox.r + padding_pt,
            t=bbox.t - padding_pt,  # ← move up   (decrease t)
            b=bbox.b + padding_pt,  # ← move down (increase b)
            coord_origin=CoordOrigin.TOPLEFT,
        )
        # ── 2. Clamp to page bounds ───────────────────────────────────────────
        clamped = BoundingBox(
            l=max(0.0, padded.l),
            r=min(page_w, padded.r),
            t=max(0.0, padded.t),
            b=min(page_h, padded.b),
            coord_origin=CoordOrigin.TOPLEFT,
        )
        # ── 3. Convert to BOTTOMLEFT — pypdfium2 uses PDF native coords ───────
        cropbox = clamped.to_bottom_left_origin(page_h)

    else:
        # BOTTOMLEFT: t > b — y increases upward
        # expand outward: grow t, shrink b
        padded = BoundingBox(
            l=bbox.l - padding_pt,
            r=bbox.r + padding_pt,
            t=bbox.t + padding_pt,  # ← move up   (increase t)
            b=bbox.b - padding_pt,  # ← move down (decrease b)
            coord_origin=CoordOrigin.BOTTOMLEFT,
        )
        cropbox = BoundingBox(
            l=max(0.0, padded.l),
            r=min(page_w, padded.r),
            t=min(page_h, padded.t),
            b=max(0.0, padded.b),
            coord_origin=CoordOrigin.BOTTOMLEFT,
        )

    # ── 4. Sanity check before handing to the backend ─────────────────────────
    if cropbox.r <= cropbox.l or cropbox.t <= cropbox.b:
        raise ValueError(
            f"Degenerate cropbox after clamping on page {page_no}: {cropbox}. "
            "bbox may be smaller than the padding value."
        )

    # ── 5. Crop ────────────────────────────────────────────────────────────────
    page_backend = backend.load_page(page_no - 1)  # 0-indexed
    try:
        return page_backend.get_page_image(scale=scale, cropbox=cropbox)
    finally:
        page_backend.unload()


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — LLM call: base64-encode crop → vision API
# ─────────────────────────────────────────────────────────────────────────────
def pil_to_base64(img: Image.Image, fmt: str = "PNG") -> str:
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode()


def reparse_with_llm(
    crop: Image.Image,
    label: str,
    model: str = VISION_MODEL,
    retries: int = RETRY_ATTEMPTS,
) -> str:
    prompt = PROMPT_MAP.get(label.upper(), DEFAULT_PROMPT)
    b64 = pil_to_base64(crop)

    payload = {
        "model": model,
        "max_tokens": 1500,
        "temperature": 0.0,  # deterministic — we want transcription not creativity
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                    {"type": "text", "text": prompt},
                ],
            }
        ],
    }

    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                    "Content-Type": "application/json",
                },
                json=payload,
                timeout=120,
            )
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()
        except Exception as e:
            if attempt == retries:
                raise
            print(f"      retry {attempt}/{retries} after error: {e}")
            time.sleep(RETRY_DELAY_S)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Dataclass to hold each reparsed element
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class ReparsedElement:
    page_no: int
    label: str
    conf: float
    bbox: BoundingBox
    llm_text: str
    crop: Image.Image | None = field(default=None, repr=False)

In [27]:
print(len(low_conf))
low_conf[:5]

10


[{'page': 1,
  'label': 'text',
  'confidence': 0.5367892384529114,
  'bbox': BoundingBox(l=394.577, t=142.82297960000005, r=547.8135040175999, b=147.3659252, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>)},
 {'page': 1,
  'label': 'key_value_region',
  'confidence': 0.5522691011428833,
  'bbox': BoundingBox(l=52.90776824951172, t=701.7467651367188, r=166.88525390625, b=709.1648559570312, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>)},
 {'page': 5,
  'label': 'list_item',
  'confidence': 0.5777231454849243,
  'bbox': BoundingBox(l=59.669000000000004, t=255.88442239999983, r=98.50459040000001, b=260.04483199999993, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>)},
 {'page': 5,
  'label': 'list_item',
  'confidence': 0.5557041168212891,
  'bbox': BoundingBox(l=56.426, t=332.5964224, r=98.50459040000001, b=336.756832, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>)},
 {'page': 5,
  'label': 'list_item',
  'confidence': 0.5672545433044434,
  'bbox': BoundingBox(l=69.041, t=352.2113755, r=21

In [33]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Main reparsing loop
# ─────────────────────────────────────────────────────────────────────────────
reparsed: list[ReparsedElement] = []

print(f"\n── Reparsing {len(low_conf)} low-confidence elements ──")

for row in low_conf:
    label = row["label"]
    page_no = row["page"]
    conf = row["confidence"]
    bbox = row["bbox"]

    print(f"  [{page_no:>2}] {label:<22} conf={conf:.3f} → ", end="", flush=True)

    try:
        page_size = page_size_map[page_no]  # ← pass size in
        crop = crop_cluster_from_backend(backend, page_no, bbox, page_size)
    except Exception as e:
        print(f"crop failed: {e}")
        continue

    try:
        llm_text = reparse_with_llm(crop, label)
        reparsed.append(
            ReparsedElement(
                page_no=page_no,
                label=label,
                conf=conf,
                bbox=bbox,
                llm_text=llm_text,
                crop=crop,
            )
        )
        print(f"✓ ({len(llm_text)} chars)")
    except Exception as e:
        print(f"LLM failed: {e}")

# backend.close()


── Reparsing 10 low-confidence elements ──
  [ 1] text                   conf=0.537 → ✓ (37 chars)
  [ 1] key_value_region       conf=0.552 → ✓ (73 chars)
  [ 5] list_item              conf=0.578 → ✓ (10 chars)
  [ 5] list_item              conf=0.556 → ✓ (11 chars)
  [ 5] list_item              conf=0.567 → ✓ (130 chars)
  [ 5] text                   conf=0.503 → ✓ (136 chars)
  [ 5] list_item              conf=0.585 → ✓ (102 chars)
  [ 5] list_item              conf=0.575 → ✓ (44 chars)
  [ 5] text                   conf=0.524 → ✓ (11 chars)
  [10] formula                conf=0.501 → ✓ (18 chars)


In [35]:
reparsed[:5]

[ReparsedElement(page_no=1, label='text', conf=0.5367892384529114, bbox=BoundingBox(l=394.577, t=142.82297960000005, r=547.8135040175999, b=147.3659252, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), llm_text='Strategic Data Solutions (SDS) Group,'),
 ReparsedElement(page_no=1, label='key_value_region', conf=0.5522691011428833, bbox=BoundingBox(l=52.90776824951172, t=701.7467651367188, r=166.88525390625, b=709.1648559570312, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), llm_text='ACM ISBN 978-1-4503-9385-0/22/08.\nhttps://doi.org/10.1145/3534678.3539108'),
 ReparsedElement(page_no=5, label='list_item', conf=0.5777231454849243, bbox=BoundingBox(l=59.669000000000004, t=255.88442239999983, r=98.50459040000001, b=260.04483199999993, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), llm_text='9: end for'),
 ReparsedElement(page_no=5, label='list_item', conf=0.5557041168212891, bbox=BoundingBox(l=56.426, t=332.5964224, r=98.50459040000001, b=336.756832, coord_origin=<CoordOrigin.TOPLEFT: 'T

In [36]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 — RAG integration: merge into document chunks
#
#  Strategy: build a bbox-keyed lookup of reparsed text.
#  When chunking, check each chunk's provenance bbox against the lookup —
#  if a match exists, substitute LLM text for the Docling-parsed text.
# ─────────────────────────────────────────────────────────────────────────────
def bbox_key(page_no: int, bbox: BoundingBox, precision: int = 1) -> str:
    """Stable string key for bbox matching (rounded to avoid float jitter)."""
    return (
        f"{page_no}:"
        f"{bbox.l:.{precision}f},{bbox.t:.{precision}f},"
        f"{bbox.r:.{precision}f},{bbox.b:.{precision}f}"
    )


reparsed_lookup: dict[str, ReparsedElement] = {bbox_key(el.page_no, el.bbox): el for el in reparsed}

In [37]:
reparsed_lookup

{'1:394.6,142.8,547.8,147.4': ReparsedElement(page_no=1, label='text', conf=0.5367892384529114, bbox=BoundingBox(l=394.577, t=142.82297960000005, r=547.8135040175999, b=147.3659252, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), llm_text='Strategic Data Solutions (SDS) Group,'),
 '1:52.9,701.7,166.9,709.2': ReparsedElement(page_no=1, label='key_value_region', conf=0.5522691011428833, bbox=BoundingBox(l=52.90776824951172, t=701.7467651367188, r=166.88525390625, b=709.1648559570312, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), llm_text='ACM ISBN 978-1-4503-9385-0/22/08.\nhttps://doi.org/10.1145/3534678.3539108'),
 '5:59.7,255.9,98.5,260.0': ReparsedElement(page_no=5, label='list_item', conf=0.5777231454849243, bbox=BoundingBox(l=59.669000000000004, t=255.88442239999983, r=98.50459040000001, b=260.04483199999993, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), llm_text='9: end for'),
 '5:56.4,332.6,98.5,336.8': ReparsedElement(page_no=5, label='list_item', conf=0.5557041168212891, bbo

In [39]:
next(iter(result.document.iterate_items()))

(SectionHeaderItem(self_ref='#/texts/1', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.SECTION_HEADER: 'section_header'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=60.611, t=707.7715948, r=551.3876231999999, b=672.5853538556999, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 83))], source=[], comments=[], orig='CausalMTA: Eliminating the User Confounding Bias for Causal Multi-touch Attribution', text='CausalMTA: Eliminating the User Confounding Bias for Causal Multi-touch Attribution', formatting=None, hyperlink=None, level=1),
 1)

In [ ]:
from docling_core.types.doc import PictureItem, TableItem  # these are stable


def _build_ref_lookup(doc) -> dict:
    """
    Build a self_ref → item dict from all items in the document.
    Used to resolve RefItems (captions, footnotes, etc.) without
    importing RefItem directly — its module path changes across versions.
    """
    lookup = {}
    for item, _ in doc.iterate_items():
        self_ref = getattr(item, "self_ref", None)
        if self_ref:
            lookup[self_ref] = item
    return lookup


def get_item_text(item, doc, _ref_lookup: dict | None = None) -> str | None:
    # ── TableItem ─────────────────────────────────────────────────────────────
    if isinstance(item, TableItem):
        try:
            return item.export_to_markdown(doc)
        except TypeError:
            return item.export_to_markdown()

    # ── PictureItem ───────────────────────────────────────────────────────────
    if isinstance(item, PictureItem):
        # 1. VLM description from your pipeline (lives in item.annotations)
        for ann in item.annotations or []:
            text = getattr(ann, "text", None)
            if text:
                return text

        # 2. Captions — each caption is a RefItem with a .cref like "#/texts/30"
        #    Resolve via the lookup dict instead of importing RefItem
        if item.captions and _ref_lookup is not None:
            caption_texts = []
            for cap_ref in item.captions:
                cref = getattr(cap_ref, "cref", None)
                if not cref:
                    continue
                cap_item = _ref_lookup.get(cref)
                if cap_item:
                    text = getattr(cap_item, "text", None)
                    if text:
                        caption_texts.append(text)
            if caption_texts:
                return " ".join(caption_texts)

        # 3. Markdown fallback — drop empty placeholder comments
        try:
            md = item.export_to_markdown(doc)
            return None if not md or md.strip().startswith("<!--") else md
        except Exception:
            return None

    # ── All other types (TextItem, SectionHeaderItem, ListItem …) ─────────────
    return getattr(item, "text", None)


def build_enhanced_chunks(result, reparsed_lookup: dict) -> list[dict]:
    chunks = []
    doc = result.document
    ref_lookup = _build_ref_lookup(doc)  # build once, reuse per PictureItem

    for item, _ in doc.iterate_items():
        if not hasattr(item, "prov") or not item.prov:
            continue

        prov = item.prov[0]
        page_no = prov.page_no
        key = bbox_key(page_no, prov.bbox)
        match = reparsed_lookup.get(key)

        print(key, match)

        if match:
            text = match.llm_text
            source = "llm_reparse"
            conf = match.conf
        else:
            text = get_item_text(item, doc, _ref_lookup=ref_lookup)
            source = "docling"
            conf = 0.99

        if not text or not text.strip():
            continue

        chunks.append(
            {
                "text": text,
                "page_no": page_no,
                "label": item.label.value if hasattr(item.label, "value") else str(item.label),
                "source": source,
                "parse_conf": round(conf, 3),
                "bbox": {
                    "l": prov.bbox.l,
                    "t": prov.bbox.t,
                    "r": prov.bbox.r,
                    "b": prov.bbox.b,
                },
            }
        )

    return chunks

In [55]:
chunks = build_enhanced_chunks(result, reparsed_lookup)
chunks

1:60.6,707.8,551.4,672.6 None
1:124.5,664.9,159.7,656.6 None
1:70.4,649.2,215.4,608.8 None
1:194.4,567.8,253.4,562.3 None
1:147.9,554.9,301.1,514.5 None
1:445.9,662.0,495.4,656.6 None
1:394.6,649.2,547.8,644.6 None
1:416.7,637.2,524.6,608.8 None
1:358.9,570.7,416.6,562.3 None
1:316.1,554.9,461.1,514.5 None
1:318.0,499.6,379.8,494.5 None
1:318.0,484.9,558.2,469.9 None
1:317.7,459.0,404.7,455.3 None
1:317.6,449.0,568.3,405.5 None
1:318.0,388.3,420.7,383.2 None
1:318.0,373.7,559.7,292.9 None
1:317.6,286.0,559.7,106.6 None
1:318.0,99.7,558.2,84.6 None
1:53.8,499.6,111.9,494.5 None
1:53.5,484.9,295.6,228.8 None
1:53.8,211.4,134.8,206.3 None
1:53.8,199.0,294.0,180.4 None
1:53.8,165.0,123.8,159.8 None
1:52.4,144.8,119.4,120.7 None
1:125.7,137.1,294.0,126.0 None
1:53.3,114.0,197.9,84.6 None
2:68.8,709.5,272.5,497.8 None
2:120.4,495.8,227.0,489.6 None
2:53.8,480.7,295.6,443.7 None
2:53.5,428.4,294.2,314.8 None
2:53.5,307.9,295.6,84.6 None
2:317.6,669.5,559.7,512.0 None
2:320.9,498.8,558.2,470.5

/var/folders/k5/l531tc5j2070y5w0jlhbhfcw0000gn/T/ipykernel_56544/2423215429.py:31: DeprecationWarning: Field `annotations` is deprecated; use `meta` instead.
  for ann in (item.annotations or []):
/var/folders/k5/l531tc5j2070y5w0jlhbhfcw0000gn/T/ipykernel_56544/2423215429.py:31: DeprecationWarning: Field `annotations` is deprecated; use `meta` instead.
  for ann in (item.annotations or []):
/var/folders/k5/l531tc5j2070y5w0jlhbhfcw0000gn/T/ipykernel_56544/2423215429.py:31: DeprecationWarning: Field `annotations` is deprecated; use `meta` instead.
  for ann in (item.annotations or []):
/var/folders/k5/l531tc5j2070y5w0jlhbhfcw0000gn/T/ipykernel_56544/2423215429.py:31: DeprecationWarning: Field `annotations` is deprecated; use `meta` instead.
  for ann in (item.annotations or []):
/var/folders/k5/l531tc5j2070y5w0jlhbhfcw0000gn/T/ipykernel_56544/2423215429.py:31: DeprecationWarning: Field `annotations` is deprecated; use `meta` instead.
  for ann in (item.annotations or []):


[{'text': 'CausalMTA: Eliminating the User Confounding Bias for Causal Multi-touch Attribution',
  'page_no': 1,
  'label': 'section_header',
  'source': 'docling',
  'parse_conf': 0.99,
  'bbox': {'l': 60.611,
   't': 707.7715948,
   'r': 551.3876231999999,
   'b': 672.5853538556999}},
 {'text': 'Di Yao ∗',
  'page_no': 1,
  'label': 'text',
  'source': 'docling',
  'parse_conf': 0.99,
  'bbox': {'l': 124.477,
   't': 664.9292634000001,
   'r': 159.7215811,
   'b': 656.5850896000001}},
 {'text': 'Institute of Computing Technology, Chinese Academy of Sciences China yaodi@ict.ac.cn Chang Gong Institute of Computing Technology, Chinese Academy of Sciences University of Chinese Academy of Sciences China gongchang21z@ict.ac.cn',
  'page_no': 1,
  'label': 'text',
  'source': 'docling',
  'parse_conf': 0.99,
  'bbox': {'l': 70.404,
   't': 649.1770204000001,
   'r': 215.38971779999997,
   'b': 608.7680748}},
 {'text': 'Sheng Chen',
  'page_no': 1,
  'label': 'section_header',
  'source': 'd

In [51]:
len(chunks)

239

In [50]:
# ── Summary ───────────────────────────────────────────────────────────────────
native_chunks = [c for c in chunks if c["source"] == "docling"]
llm_chunks = [c for c in chunks if c["source"] == "llm_reparse"]

print("\n── Chunk summary ──")
print(f"  Total chunks   : {len(chunks)}")
print(f"  Native (docling): {len(native_chunks)}")
print(f"  LLM reparsed   : {len(llm_chunks)}")
print("\n── LLM reparsed content ──")
for c in llm_chunks:
    print(f"  Page {c['page_no']} | {c['label']} | conf was {c['parse_conf']}")
    print(f"    {c['text'][:120]!r}{'...' if len(c['text']) > 120 else ''}\n")


── Chunk summary ──
  Total chunks   : 239
  Native (docling): 239
  LLM reparsed   : 0

── LLM reparsed content ──
